<a href="https://colab.research.google.com/github/EV-2002/-CRM-CM-Coursework/blob/main/CRM_Coursework_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Import all required libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Part 1: task 1.1 Extract S&P 500 tickers from Wikipedia using Pandas
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

tickers = sp500_df['Symbol'].tolist()
tickers = [t.replace('.', '-') for t in tickers]

print('Total tickers extracted:', len(tickers))
sp500_df[['Symbol', 'Security', 'GICS Sector']].head(10)

Total tickers extracted: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [4]:
# Task 1.1 (continued): store the ticker list and record the retrieval date
import os
from datetime import datetime, timezone

os.makedirs('data', exist_ok=True)

retrieval_date = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

sp500_df['yf_symbol'] = tickers
sp500_df['retrieval_date'] = retrieval_date
sp500_df.to_csv('data/sp500_constituents.csv', index=False)

print('Retrieval date:', retrieval_date)
print('Saved', len(sp500_df), 'constituents to data/sp500_constituents.csv')

Retrieval date: 2026-11-22 14:10 UTC
Saved 503 constituents to data/sp500_constituents.csv


In [6]:
# Part 1.1: Load daily market data via yfinance
start = '2021-01-01'
end   = '2025-01-01'   # end date is exclusive in yfinance

data = yf.download(tickers, start=start, end=end, auto_adjust=False)

close  = data['Close']        # daily (raw) closing prices
df     = data['Adj Close']    # adjusted prices (same name you used before)
volume = data['Volume']       # daily volume

# S&P 500 benchmark
gspc = yf.download('^GSPC', start=start, end=end, auto_adjust=False)['Adj Close'].squeeze()

print('Shape:', df.shape)
print('Benchmark days:', len(gspc))
print('yfinance version:', yf.__version__)
df.head()


[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'FDXF', 'Q', 'VYLR', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
[*********************100%***********************]  1 of 1 completed

Shape: (1005, 503)
Benchmark days: 1005
yfinance version: 0.2.66


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,114.009148,125.537140,84.219154,139.149994,97.269699,33.186359,232.029327,485.339996,132.622635,42.189487,...,26.124495,102.852081,54.757389,33.042767,93.015892,221.160004,95.016190,141.603363,378.130005,152.952026
2021-01-05,114.941238,127.089218,85.090019,148.300003,98.473198,33.319485,233.350281,485.690002,134.993134,43.010098,...,26.371326,106.017525,54.223648,34.635193,92.903923,229.750000,95.025169,144.053406,380.570007,154.017853
2021-01-06,118.093239,122.811218,84.354942,142.770004,98.268166,34.783871,235.901611,466.309998,135.601700,44.752831,...,26.506689,106.652512,55.424541,35.518974,96.822731,226.889999,95.545952,148.158356,394.820007,156.280518
2021-01-07,121.235588,127.001923,85.257812,151.270004,99.222061,34.460564,238.127304,477.739990,141.332718,44.355225,...,26.761478,105.594185,54.340416,35.797649,101.217415,239.479996,94.818649,147.280075,409.100006,156.140274
2021-01-08,122.100456,128.098099,85.705223,149.770004,99.498398,34.650742,238.995865,485.100006,142.359039,44.473660,...,26.578348,104.901443,54.840782,36.195755,100.191040,241.449997,96.210403,146.956451,405.470001,157.178085


In [7]:
# Task 1.2 (Step A): save a documented cached snapshot of the raw data
import json

os.makedirs('data', exist_ok=True)

close.to_csv('data/close.csv.gz')
df.to_csv('data/adj_close.csv.gz')
volume.to_csv('data/volume.csv.gz')
gspc.to_frame('^GSPC').to_csv('data/gspc.csv.gz')

failed = [t for t in tickers if df[t].dropna().empty]

metadata = {
    'constituents_source': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies',
    'constituents_retrieved': retrieval_date,
    'prices_downloaded': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
    'price_source': 'Yahoo Finance via yfinance',
    'yfinance_version': yf.__version__,
    'period_start': '2021-01-01',
    'period_end_exclusive': '2025-01-01',
    'n_tickers_listed': len(tickers),
    'n_trading_days': len(df),
    'failed_downloads': failed,
}
with open('data/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))
print('\nFiles saved:', sorted(os.listdir('data')))

{
  "constituents_source": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
  "constituents_retrieved": "2026-11-22 14:10 UTC",
  "prices_downloaded": "2026-11-22 14:23 UTC",
  "price_source": "Yahoo Finance via yfinance",
  "yfinance_version": "0.2.66",
  "period_start": "2021-01-01",
  "period_end_exclusive": "2025-01-01",
  "n_tickers_listed": 503,
  "n_trading_days": 1005,
  "failed_downloads": [
    "FDXF",
    "HONA",
    "Q",
    "SNDK",
    "VYLR"
  ]
}

Files saved: ['adj_close.csv.gz', 'close.csv.gz', 'gspc.csv.gz', 'metadata.json', 'sp500_constituents.csv', 'volume.csv.gz']


In [8]:
#downloading raw data  before cleaning
!zip -r data_snapshot.zip data
from google.colab import files
files.download('data_snapshot.zip')

  adding: data/ (stored 0%)
  adding: data/gspc.csv.gz (stored 0%)
  adding: data/close.csv.gz (deflated 0%)
  adding: data/adj_close.csv.gz (deflated 0%)
  adding: data/volume.csv.gz (deflated 0%)
  adding: data/metadata.json (deflated 41%)
  adding: data/sp500_constituents.csv (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>